Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: Apache-2.0

# Running MMSeqs2 MSA Generation on AWS Batch

## Table of Contents
0. [Install Dependencies](#0.-Install-Dependencies)
1. [Create Target](#1.-Create-Target)
2. [Submit MMSeqs2 MSA Jobs](#2.-Submit-MMSeqs2-MSA-Jobs)
3. [Download and View Results](#3.-Download-and-View-Results)

## 0. Install Dependencies

In [ ]:
%pip install -U -q -r notebook-requirements.txt

In [ ]:
# Import required Python packages

import boto3
from datetime import datetime
from batchfold.batchfold_environment import BatchFoldEnvironment
from batchfold.batchfold_target import BatchFoldTarget
from batchfold.mmseqs2_job import MMSeqs2Job

# Create AWS clients
boto_session = boto3.session.Session()

batch_environment = BatchFoldEnvironment(boto_session=boto_session)

S3_BUCKET = batch_environment.default_bucket
print(f" S3 bucket name is {S3_BUCKET}")

## 1. Create Target

In [ ]:
target_id = "7FCC"
target = BatchFoldTarget(target_id=target_id, s3_bucket=S3_BUCKET, boto_session=boto_session)
target.add_sequence(
    seq_id=target_id,
    seq="KEYDIYVSYARNAEEEEFVLLTLRGVLENEFGYKLCIFDRDSLPGGNTVEAVFDFIQRSRRMIVVLSPDYVTEKSISMLEFKLGVMCQNSIATKLIVVEYRPLEHPHPGILQLKESVSFVSWKGEKSKHSGSKFWKALRLALPLRS",
    description="Chain A|Isoform 4 of Interleukin-1 receptor accessory protein|Homo sapiens (9606)",
)

## 2. Submit MMSeqs2 MSA Jobs

List available job queues

In [ ]:
batch_environment.list_job_queue_names()

### Submit MMSeqs2 CPU job

This submits a CPU-based MMSeqs2 MSA generation job searching against UniRef90.

In [ ]:
job_name = target.target_id + "_MMSeqs2Job_" + datetime.now().strftime("%Y%m%d%s")
mmseqs2_job = MMSeqs2Job(
    job_name=job_name,
    target_id=target.target_id,
    fasta_s3_uri=target.get_fasta_s3_uri(),
    output_s3_uri=target.get_msas_s3_uri(),
    boto_session=boto_session,
    databases=["uniref90"],
    databases_paths_overwrite={
        "uniref90": "mmseqs2_dbs/uniref90/uniref90",
    },
    cpu=16,
    memory=31,
)
mmseqs2_submission = batch_environment.submit_job(
    mmseqs2_job, job_queue_name="CPUOnDemandJobQueue"
)

### Submit MMSeqs2 GPU job (optional)

MMSeqs2 also supports GPU-accelerated search. Set `gpu=1` to use the GPU container.

In [ ]:
job_name = target.target_id + "_MMSeqs2GPUJob_" + datetime.now().strftime("%Y%m%d%s")
mmseqs2_gpu_job = MMSeqs2Job(
    job_name=job_name,
    target_id=target.target_id,
    fasta_s3_uri=target.get_fasta_s3_uri(),
    output_s3_uri=target.get_msas_s3_uri(),
    boto_session=boto_session,
    databases=["uniref90"],
    databases_paths_overwrite={
        "uniref90": "mmseqs2_dbs/uniref90/uniref90",
    },
    cpu=8,
    memory=31,
    gpu=1,
)
mmseqs2_gpu_submission = batch_environment.submit_job(
    mmseqs2_gpu_job, job_queue_name="G4dnJobQueue"
)

### Multi-database search example

MMSeqs2 supports searching against multiple databases in a single job. Results are merged into a combined MSA.

In [ ]:
job_name = target.target_id + "_MMSeqs2MultiDBJob_" + datetime.now().strftime("%Y%m%d%s")
mmseqs2_multidb_job = MMSeqs2Job(
    job_name=job_name,
    target_id=target.target_id,
    fasta_s3_uri=target.get_fasta_s3_uri(),
    output_s3_uri=target.get_msas_s3_uri(),
    boto_session=boto_session,
    databases=["uniref90", "swissprot"],
    databases_paths_overwrite={
        "uniref90": "mmseqs2_dbs/uniref90/uniref90",
        "swissprot": "mmseqs2_dbs/swissprot/swissprot",
    },
    evalue=0.001,
    sensitivity=7.5,
    num_iterations=3,
    cpu=16,
    memory=31,
)
mmseqs2_multidb_submission = batch_environment.submit_job(
    mmseqs2_multidb_job, job_queue_name="CPUOnDemandJobQueue"
)

### Check on job statuses

In [ ]:
for job in [mmseqs2_job]:
    print(
        f"Job {job.describe_job()[0]['jobName']} is in status {job.describe_job()[0]['status']}"
    )

## 3. Download and View Results

Once the jobs are finished, download and view the MSA results

In [ ]:
target = BatchFoldTarget(
    target_id=target_id, s3_bucket=S3_BUCKET, boto_session=boto_session
)

In [ ]:
target.download_msas(local_path="data")

### View MSA output files

MMSeqs2 generates the following output files:
- `mmseqs_output.a3m` - Combined MSA in A3M format
- `mmseqs_hits_combined.tsv` - Combined hits table across all databases
- `mmseqs_hits_<db>.tsv` - Per-database hits tables
- `mmseqs_summary.json` - Summary statistics (hit counts, avg identity, etc.)
- `mmseqs_config.json` - Resolved job parameters
- `mmseqs_logs.log` - Execution logs

In [ ]:
import os
msa_dir = f"data/{target_id}/msas/mmseqs2"
if os.path.exists(msa_dir):
    for f in os.listdir(msa_dir):
        filepath = os.path.join(msa_dir, f)
        size = os.path.getsize(filepath)
        print(f"  {f} ({size:,} bytes)")
else:
    print(f"MSA directory not found at {msa_dir}. Check that the job has completed.")